# API wrappers

The OpenWeatherMap offers REST endpoints for querying current weather, forecasts, historical data, etc. However, accessing this data directly via the REST API requires handling multiple API calls, query parameters, and response parsing. The pyowm library abstracts these complexities and provides useful built-in functionalities.

After signing in to OpenWeatherMap retrieve your api key at https://home.openweathermap.org/api_keys

You will also need to install the pyowm package: pip install pyowm 

In [39]:
import requests
import pyowm
import json

api_key = '9a8c09798afc0194220311045d7a5b64'

## use case 1: managing API keys

In a raw rest API call you always have to manage credentials in each individual call. Wrappers usually store and manage the authentication for you

In [40]:
#You can get current weather data by making a GET request to an endpoint like:

params = {
    'appid' : api_key
}

response = requests.get('https://api.openweathermap.org/data/2.5/weather?q=London', params = params)

json.loads(response.text)

#but for every call you make using GET from now on you do need to add the parameters, since the raw API does not manage authentication for you

{'coord': {'lon': -0.1257, 'lat': 51.5085},
 'weather': [{'id': 803,
   'main': 'Clouds',
   'description': 'broken clouds',
   'icon': '04n'}],
 'base': 'stations',
 'main': {'temp': 278.66,
  'feels_like': 275.87,
  'temp_min': 276.99,
  'temp_max': 279.31,
  'pressure': 999,
  'humidity': 76,
  'sea_level': 999,
  'grnd_level': 995},
 'visibility': 10000,
 'wind': {'speed': 3.6, 'deg': 230},
 'clouds': {'all': 73},
 'dt': 1773434382,
 'sys': {'type': 2,
  'id': 2075535,
  'country': 'GB',
  'sunrise': 1773382779,
  'sunset': 1773424834},
 'timezone': 0,
 'id': 2643743,
 'name': 'London',
 'cod': 200}

Most wrappers (pyowm included) include some way of initializing a session with the authentication key that you then don't need to type again.

Initialize pyowm with the default configuration. Thenopen the weather manager

Check out a snippet here: https://pyowm.readthedocs.io/en/latest/v3/code-recipes.html#weather_data

In [41]:
#your code here
from pyowm import OWM
owm = OWM(api_key)
mgr = owm.weather_manager()

## use case 2: Simplified calls

With the raw REST API, you'd have to build a URL manually, send the request, and parse the JSON response to get the current weather.

In [42]:
city = 'London'
url = f'http://api.openweathermap.org/data/2.5/weather?q={city}'

response = requests.get(url,params= params)
data = response.json()
temperature = data['main']['temp']
humidity = data['main']['humidity']
wind_speed = data['wind']['speed']

print(f"Temperature: {temperature}°C, Humidity: {humidity}%, Wind Speed: {wind_speed} m/s")

Temperature: 278.66°C, Humidity: 76%, Wind Speed: 3.6 m/s


Get the equivalent call as above for the city of London using the pyowm package

In [54]:
#your code here

observation = mgr.weather_at_place('London').weather
temp_dict_kelvin = observation.temperature()
temperature = temp_dict_kelvin['temp']
wind_dict_in_meters_per_sec = observation.wind()   # Default unit: 'meters_sec'
wind_speed = wind_dict_in_meters_per_sec['speed']
humidity = observation.humidity
print(f"Temperature: {temperature}°C, Humidity: {humidity}%, Wind Speed: {wind_speed} m/s")

Temperature: 278.18°C, Humidity: 77%, Wind Speed: 5.14 m/s


## use case 3: Combining and chaining calls

Wrappers often offer methods that make multiple calls to batch requests that make sense to batch. And often they offer methods that make sequences of calls that each returns information necessary to make the next call.

For example, to get a weather forecast for a specific city using the raw API you need to first geocode the city to get its latitude and longitude:

In [47]:
city = 'New York'
geocode_url = f'http://api.openweathermap.org/data/2.5/weather?q={city}'
geocode_response = requests.get(geocode_url,params=params).json()

lat = geocode_response['coord']['lat']
lon = geocode_response['coord']['lon']

Then, request the weather forecast for that latitude/longitude:

In [48]:
forecast_url = f'http://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}'
forecast_response = requests.get(forecast_url, params=params).json()

for entry in forecast_response['list']:
    print(f"Time: {entry['dt_txt']}, Temp: {entry['main']['temp']}°C")

Time: 2026-03-14 00:00:00, Temp: 278.23°C
Time: 2026-03-14 03:00:00, Temp: 279.62°C
Time: 2026-03-14 06:00:00, Temp: 281.06°C
Time: 2026-03-14 09:00:00, Temp: 280.44°C
Time: 2026-03-14 12:00:00, Temp: 279.85°C
Time: 2026-03-14 15:00:00, Temp: 278.65°C
Time: 2026-03-14 18:00:00, Temp: 281.54°C
Time: 2026-03-14 21:00:00, Temp: 282.25°C
Time: 2026-03-15 00:00:00, Temp: 279.92°C
Time: 2026-03-15 03:00:00, Temp: 277.4°C
Time: 2026-03-15 06:00:00, Temp: 276.13°C
Time: 2026-03-15 09:00:00, Temp: 275.43°C
Time: 2026-03-15 12:00:00, Temp: 275.53°C
Time: 2026-03-15 15:00:00, Temp: 278.11°C
Time: 2026-03-15 18:00:00, Temp: 279.11°C
Time: 2026-03-15 21:00:00, Temp: 279.8°C
Time: 2026-03-16 00:00:00, Temp: 279.7°C
Time: 2026-03-16 03:00:00, Temp: 280.76°C
Time: 2026-03-16 06:00:00, Temp: 281.27°C
Time: 2026-03-16 09:00:00, Temp: 282.02°C
Time: 2026-03-16 12:00:00, Temp: 282.13°C
Time: 2026-03-16 15:00:00, Temp: 283.72°C
Time: 2026-03-16 18:00:00, Temp: 284.68°C
Time: 2026-03-16 21:00:00, Temp: 285.

Two calls: one for geocoding, one for forecasts.
But with pyowm, because this is a common operation, there is a method that handles the geocoding internally and then fetches the weather forecast in one step.

Get the above forecast in a single call using pyowm.

Hint: search for "forecast_at_place" in the code recipies of the documentation

In [ ]:
#your code here
forecast = mgr.forecast_at_place('New York,US', '3h').forecast

for weather in forecast:
    print(
        f"Time: {weather.reference_time('iso')}, "
        f"Temp: {weather.temperature('celsius')['temp']}°C"
    )


Time: 2026-03-14 00:00:00+00:00, Temp: 5.07°C
Time: 2026-03-14 03:00:00+00:00, Temp: 6.46°C
Time: 2026-03-14 06:00:00+00:00, Temp: 7.91°C
Time: 2026-03-14 09:00:00+00:00, Temp: 7.29°C
Time: 2026-03-14 12:00:00+00:00, Temp: 6.7°C
Time: 2026-03-14 15:00:00+00:00, Temp: 5.5°C
Time: 2026-03-14 18:00:00+00:00, Temp: 8.39°C
Time: 2026-03-14 21:00:00+00:00, Temp: 9.1°C
Time: 2026-03-15 00:00:00+00:00, Temp: 6.77°C
Time: 2026-03-15 03:00:00+00:00, Temp: 4.25°C
Time: 2026-03-15 06:00:00+00:00, Temp: 2.98°C
Time: 2026-03-15 09:00:00+00:00, Temp: 2.28°C
Time: 2026-03-15 12:00:00+00:00, Temp: 2.38°C
Time: 2026-03-15 15:00:00+00:00, Temp: 4.96°C
Time: 2026-03-15 18:00:00+00:00, Temp: 5.96°C
Time: 2026-03-15 21:00:00+00:00, Temp: 6.65°C
Time: 2026-03-16 00:00:00+00:00, Temp: 6.55°C
Time: 2026-03-16 03:00:00+00:00, Temp: 7.61°C
Time: 2026-03-16 06:00:00+00:00, Temp: 8.12°C
Time: 2026-03-16 09:00:00+00:00, Temp: 8.87°C
Time: 2026-03-16 12:00:00+00:00, Temp: 8.98°C
Time: 2026-03-16 15:00:00+00:00, Temp

## use case 4: Convenience methods

Wrappers often offer built-in methods to handle common kinds of tasks related to the APIs, reducing the need for manual calculations.

for example converting units (e.g., temperature from Celsius to Fahrenheit) or working with more complex data requires manual conversion when using the raw API.

In [60]:
city = 'London'
url = f'http://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}'

response = requests.get(url)
data = response.json()
temperature_celsius = data['main']['temp']
temperature_fahrenheit = (temperature_celsius * 9/5) + 32

print(f"Temperature in Celsius: {temperature_celsius}°C, Fahrenheit: {temperature_fahrenheit}°F")

Temperature in Celsius: 278.1°C, Fahrenheit: 532.58°F


But the pyowm wrapper offers built-in methods to handle these kinds of tasks, reducing the need for manual calculations.
Get the temperature both in Celcius and Farenheit using pyowm. Navigate the code recipes to figure out the inbuilt methods for this.

In [59]:
#your code here
weather = mgr.weather_at_place('London').weather
temp_dict_kelvin = weather.temperature()   # a dict in Kelvin units (default when no temperature units provided)

temp_dict_fahrenheit = weather.temperature('fahrenheit')  # a dict in Fahrenheit units
fah = temp_dict_fahrenheit['temp']
temp_dict_celsius = weather.temperature('celsius')  # guess?
cel = temp_dict_celsius['temp']
print(f"Temperature in Celsius: {cel}°C, Fahrenheit: {fah}°F")

Temperature in Celsius: 4.92°C, Fahrenheit: 40.86°F
